# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walk-through for loading and exploring a dataset published in Croissant format using the `mlcroissant` Python library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset from Croissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

print(f"Published: {metadata.datePublished}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")
print(f"Version: {getattr(metadata, 'version', '')}")

## 2. Data Overview
Review available record sets and their field `@id`s. All entity references use their `@id` as specified in the Croissant schema.

In [ ]:
# List available record sets by @id
if hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
    if isinstance(record_sets, dict):
        record_sets = [record_sets]
    print("Record sets and available fields (referenced by @id):\n")
    for rs in record_sets:
        print(f"- Record set '@id': {getattr(rs, '@id', None)}; Name: {getattr(rs, 'name', '')}")
        if hasattr(rs, 'field'):
            fields = rs.field
            if isinstance(fields, dict):
                fields = [fields]
            for field in fields:
                print(f"  - Field '@id': {getattr(field, '@id', None)}; Name: {getattr(field, 'name', '')}; Data type: {getattr(field, 'dataType', '')}")
else:
    print("No record sets available in the metadata.")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Note:** If the dataset contains multiple record sets, adapt the extraction for each; otherwise, focus on the main one.

In [ ]:
# Get all record sets by @id
def get_recordset_ids(metadata):
    record_sets = getattr(metadata, 'recordSet', [])
    if isinstance(record_sets, dict):
        record_sets = [record_sets]
    record_set_ids = []
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        if rs_id:
            record_set_ids.append(rs_id)
    return record_set_ids

record_set_ids = get_recordset_ids(metadata)
print(f"Extracting data from record sets: {record_set_ids}")

dataframes = {}
for record_set_id in record_set_ids:
    # Use the @id as required by mlcroissant
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nFirst five columns of record set {record_set_id}:")
    print(df.columns.tolist())
    print(df.head())

# Select the primary record set for further analysis (if there is only one, use it)
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps: filter records, normalize numeric fields, group by attributes, and summarize data. All steps demonstrate referencing by `@id` and variable use for flexibility.

In [ ]:
if main_record_set_id is not None:
    df = dataframes[main_record_set_id].copy()
    # Attempt to detect numeric columns
    numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    print(f"Numeric fields: {numeric_fields}")

    # Example: Use first numeric column (could be 'coverage', 'MW', etc.; adapt as appropriate)
    if len(numeric_fields) > 0:
        numeric_field_id = numeric_fields[0]
        # Choose a sensible threshold
        threshold = df[numeric_field_id].quantile(0.5)
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt grouping by a likely group field
        possible_group_fields = [c for c in df.columns if c.lower() in ['sample', 'category', 'group', 'condition', 'description']]
        group_field = possible_group_fields[0] if len(possible_group_fields) > 0 else None

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No valid categorical field found for grouping.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No main record set to analyze.")

## 5. Visualization
Visualize data distributions and relationships using Matplotlib and Seaborn.

In [ ]:
if main_record_set_id is not None and len(numeric_fields) > 0:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    # If a group_field is available, plot boxplots
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore a Croissant dataset with `mlcroissant`. We programmatically listed record sets and referenced each entity by its `@id`, loaded data into DataFrames, conducted EDA including filtering and normalization, and visualized results.

**You can adapt and extend this notebook for downstream analyses, including machine learning models or detailed biological investigation.**
